In [ ]:
suppressPackageStartupMessages({
    library(ggplot2)
    library(Seurat)
    library(dplyr)
    library(tidyr)
    library(tibble)
    library(stringr)
    library(dittoSeq)
    library(patchwork)
    library(future)
    plan("multicore", workers = 12)
    options(future.globals.maxSize = 1000 * 1024^5)
    options(stringsAsFactors = FALSE)
    set.seed(123)
})

In [ ]:
path_ref <- '/projects/0/einf2548/cruiz/dmg/data/rna_dmg_atlas_scglue_embbeding.rds'
reference <- readRDS(path_ref)
reference

An object of class Seurat 
19248 features across 397794 samples within 1 assay 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 2 dimensional reductions calculated: pca, umap

In [ ]:
reference <- AddMetaData(reference, readRDS('../data/dmg_atlas_final_annotation.rds'))

In [ ]:
desisto2023 <- readRDS('/projects/0/einf2548/cruiz/dmg/data/DeSisto_qc_filtered_and_normalized.rds')
desisto2023

An object of class Seurat 
16193 features across 6979 samples within 1 assay 
Active assay: RNA (16193 features, 0 variable features)
 1 layer present: data
 2 dimensional reductions calculated: pca, umap

In [ ]:
desisto2023 <-FindVariableFeatures(desisto2023)%>%ScaleData()%>%RunPCA(verbose = FALSE)%>%
        FindNeighbors(verbose = F) %>% FindClusters(verbose = F)%>%
        RunUMAP(dims = 1:15, reduction = "pca", verbose = F)

Warning message in FindVariableFeatures.Assay(object = object[[assay]], selection.method = selection.method, :
“selection.method set to 'vst' but count slot is empty; will use data slot instead”
Centering and scaling data matrix

Warning message:
“Number of dimensions changing from 15 to 50”
Warning message:
“UNRELIABLE VALUE: One of the ‘future.apply’ iterations (‘future_lapply-1’) unexpectedly generated random numbers without declaring so. There is a risk that those random numbers are not statistically sound and the overall results might be invalid. To fix this, specify 'future.seed=TRUE'. This ensures that proper, parallel-safe random numbers are produced via the L'Ecuyer-CMRG method. To disable this check, use 'future.seed = NULL', or set option 'future.rng.onMisuse' to "ignore".”
Warning message:
“The default method for RunUMAP has changed from calling Python UMAP via reticulate to the R-native UWOT using the cosine metric
To use Python UMAP via reticulate, set umap.method to 'uma

### Reference mapping

In [ ]:
anchors <- FindTransferAnchors(
  reference = reference,
  query = desisto2023,
  features = rownames(reference[["RNA"]]),
  normalization.method = "LogNormalize",
  reference.reduction = "pca",
  dims = 1:50
)

Warning message:
“3055 features of the features specified were not present in both the reference query assays. 
Continuing with remaining 16193 features.”
Projecting cell embeddings

Finding neighborhoods

Finding anchors

	Found 6151 anchors



`Map Query` functions run separately

In [ ]:
desisto2023 <- TransferData(
  anchorset = anchors, 
  reference = reference,
  query = desisto2023,
  refdata = list(
      dmg.atlas.lvl_1 = 'lvl_1',
      dmg.atlas.lvl_2 = 'lvl_2',
      dmg.atlas.lvl_3 = 'lvl_3',
      dmg.atlas.lvl_4 = 'lvl_4',
      dmg.atlas.lvl_5 = 'lvl_5'
  ),
)

Finding integration vectors

Finding integration vector weights

Predicting cell labels

Predicting cell labels

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Predicting cell labels

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Predicting cell labels

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”
Predicting cell labels

Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


In [ ]:
desisto2023 <- IntegrateEmbeddings(
  anchorset = anchors,
  reference = reference,
  query = desisto2023, 
  new.reduction.name = "ref.pca"
)


Integrating dataset 2 with reference dataset

Finding integration vectors

Integrating data



### UMAP embbeding projection

In [ ]:
desisto2023 <- ProjectUMAP(
  query = desisto2023, 
  query.reduction = "ref.pca", 
  reference = reference, 
  reference.reduction = "pca", 
  reduction.model = "umap"
)

Computing nearest neighbors

Running UMAP projection

17:08:06 Read 6979 rows

17:08:06 Processing block 1 of 1

17:08:06 Commencing smooth kNN distance calibration using 12 threads
 with target n_neighbors = 30

17:08:06 Initializing by weighted average of neighbor coordinates using 12 threads

17:08:06 Commencing optimization for 67 epochs, with 209370 positive edges

17:08:08 Finished



In [ ]:
saveRDS(desisto2023, '../data/DeSisto_dataset_pDG_atlas_predictions.rds')